# Constructing AutoTPD+

AutoTPD+ was constructed by merging AutoTPD with MolGlueDB or PROTAC-DB 3.0 at the DOI level. For papers covered by AutoTPD, the original database records were replaced with standardized AutoTPD records; records from papers outside the AutoTPD extraction scope were retained.

In [1]:
import pandas as pd
import numpy as np

df_autotpd_protac = pd.read_csv('/Users/yaochenr/project/tpd_curator/data/release/AutoTPD-PROTAC.csv')
df_autotpd_mg = pd.read_csv('/Users/yaochenr/project/tpd_curator/data/release/AutoTPD-MG.csv')

df_protacdb = pd.read_csv('/Users/yaochenr/project/tpd_curator/data/evaluation/protacs/baseline/PROTACDB-baseline-cleaned-std.csv')
df_molgluedb = pd.read_csv('/Users/yaochenr/project/tpd_curator/data/evaluation/molecular_glues/baseline/MolGlueDB-baseline-cleaned-std.csv')

## Standarize MolGlueDB and PROTAC-DB

In [2]:
def normalize_doi(doi_series):
    return (
        doi_series.astype('string')
        .str.strip()
        .str.lower()
        .str.replace(r'^https?://(dx\.)?doi\.org/', '', regex=True)
        .str.replace(r'^doi:', '', regex=True)
        .str.strip()
    )


def merge_by_doi_replace_append(baseline_df, autotpd_df, baseline_name, autotpd_name, doi_col='DOI'):
    baseline_dois_normalized = normalize_doi(baseline_df[doi_col])
    autotpd_dois_normalized = normalize_doi(autotpd_df[doi_col]).dropna().unique()
    rows_to_replace = baseline_dois_normalized.isin(autotpd_dois_normalized)

    merged_df = pd.concat(
        [
            baseline_df[~rows_to_replace],
            autotpd_df,
        ],
        ignore_index=True,
        sort=False,
    )
    print(f"{baseline_name} original rows: {len(baseline_df)}")
    print(f"{autotpd_name} original rows: {len(autotpd_df)}")
    print(f"Merged rows: {len(merged_df)}")
    return merged_df


df_autotpd_protac_for_merge = df_autotpd_protac.rename(
    columns={'Standard_InChIKey': 'InChIKey'}
)


df_protac_merged = merge_by_doi_replace_append(
    baseline_df=df_protacdb,
    autotpd_df=df_autotpd_protac_for_merge,
    baseline_name='PROTACDB',
    autotpd_name='AutoTPD PROTAC',
)

df_protac_merged

PROTACDB original rows: 2219
AutoTPD PROTAC original rows: 714
Merged rows: 2532


,Record_ID,Compound_ID,Compound_Name,SMILES,Compound_Key,InChI,InChIKey,Degradation_Target,Degradation_Target_Gene,Degradation_Target_Uniprot,...,Dmax_conc_range_min,Dmax_conc_range_max,Dmax_conc_units,DOI,Connectivity_Key,AutoTPD_ID,IUPAC_Name,IUPAC_Name_Source,SMILES_Source,Standard_InChIKey_Source
0,7623.0,3745.0,BD-7148,Cc1nnc2n1-c1sc(C#Cc3cnn(C4CN(c5cccc6c5C(=O)N(C...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,BRD2,P25440,...,NaN,NaN,NaN,10.1021/acs.jmedchem.3c00520,JTXHXYXYGGEYBL,NaN,NaN,NaN,NaN,NaN
1,7624.0,3745.0,BD-7148,Cc1nnc2n1-c1sc(C#Cc3cnn(C4CN(c5cccc6c5C(=O)N(C...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,BRD2,P25440,...,NaN,NaN,NaN,10.1021/acs.jmedchem.3c00520,JTXHXYXYGGEYBL,NaN,NaN,NaN,NaN,NaN
2,7625.0,3745.0,BD-7148,Cc1nnc2n1-c1sc(C#Cc3cnn(C4CN(c5cccc6c5C(=O)N(C...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,BRD2,P25440,...,NaN,NaN,NaN,10.1021/acs.jmedchem.3c00520,JTXHXYXYGGEYBL,NaN,NaN,NaN,NaN,NaN
3,7625.0,3745.0,BD-7148,Cc1nnc2n1-c1sc(C#Cc3cnn(C4CN(c5cccc6c5C(=O)N(C...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,InChI=1S/C37H30N8O5S/c1-21-40-41-31-20-50-19-2...,JTXHXYXYGGEYBL-UHFFFAOYSA-N,BRD2,BRD2,P25440,...,NaN,NaN,NaN,10.1021/acs.jmedchem.3c00520,JTXHXYXYGGEYBL,NaN,NaN,NaN,NaN,NaN
4,7647.0,3747.0,BD-9136,Cc1nnc2n1-c1sc(C#Cc3cnn(C4(CCN5CCN(C)CC5)CN(c5...,WNTMFWJFGDGXOH-UHFFFAOYSA-N,InChI=1S/C44H44N10O5S/c1-28-47-48-37-25-59-24-...,WNTMFWJFGDGXOH-UHFFFAOYSA-N,BRD2,BRD2,P25440,...,NaN,NaN,NaN,10.1021/acs.jmedchem.3c00520,WNTMFWJFGDGXOH,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2527,NaN,NaN,BTR2004,NaN,NaN,NaN,NaN,BRD3,BRD3,Q15059,...,NaN,NaN,nM,10.1101/gad.349717.122,NaN,AUTOTPD-PROTAC-000710,NaN,NaN,NaN,NaN
2528,NaN,NaN,BTR2004,NaN,NaN,NaN,NaN,BRD4,BRD4,O60885,...,NaN,NaN,nM,10.1101/gad.349717.122,NaN,AUTOTPD-PROTAC-000711,NaN,NaN,NaN,NaN
2529,NaN,NaN,FS-ARV-825,NaN,NaN,NaN,NaN,GSPT1,GSPT1,P15170,...,NaN,NaN,NaN,10.1039/d3cb00103b,NaN,AUTOTPD-PROTAC-000712,NaN,NaN,NaN,NaN
2530,NaN,NaN,ARV-825,Cc1sc2c(c1C)C(c1ccc(Cl)cc1)=N[C@@H](CC(=O)Nc1c...,RWLOGRLTDKDANT-TYIYNAFKSA-N,NaN,RWLOGRLTDKDANT-TYIYNAFKSA-N,IKZF1,IKZF1,Q13422,...,NaN,NaN,NaN,10.1039/d3cb00103b,RWLOGRLTDKDANT,AUTOTPD-PROTAC-000713,"2-[(9S)-7-(4-chlorophenyl)-4,5,13-trimethyl-3-...",PUBCHEM,PUBCHEM,PUBCHEM


In [3]:
df_autotpd_mg_for_merge = df_autotpd_mg.rename(
    columns={'Standard_InChIKey': 'StdInChIKey'}
)


df_mg_merged = merge_by_doi_replace_append(
    baseline_df=df_molgluedb,
    autotpd_df=df_autotpd_mg_for_merge,
    baseline_name='MolGlueDB',
    autotpd_name='AutoTPD MG',
)

df_mg_merged

MolGlueDB original rows: 486
AutoTPD MG original rows: 246
Merged rows: 618


,Record_ID,Compound_ID,Compound_Name,SMILES,Compound_Key,StdInChl,StdInChIKey,Degradation_Target,Degradation_Target_Gene,Degradation_Target_Uniprot,...,Dmax_conc_range_min,Dmax_conc_range_max,Dmax_conc_units,DOI,Connectivity_Key,AutoTPD_ID,IUPAC_Name,IUPAC_Name_Source,SMILES_Source,Standard_InChIKey_Source
0,810.0,747.0,FPFT-2216,COc1cscc1-c1cn(C2CCC(=O)NC2=O)nn1,SKUSCUALKIOIST-UHFFFAOYSA-N,InChI=1S/C12H12N4O3S/c1-19-10-6-20-5-7(10)8-4-...,SKUSCUALKIOIST-UHFFFAOYSA-N,CK1α,CSNK1A1,P48729,...,NaN,NaN,NaN,https://doi.org/10.1016/j.bmcl.2025.130193,SKUSCUALKIOIST,NaN,NaN,NaN,NaN,NaN
1,810.0,747.0,FPFT-2216,COc1cscc1-c1cn(C2CCC(=O)NC2=O)nn1,SKUSCUALKIOIST-UHFFFAOYSA-N,InChI=1S/C12H12N4O3S/c1-19-10-6-20-5-7(10)8-4-...,SKUSCUALKIOIST-UHFFFAOYSA-N,IKZF1,IKZF1,Q13422,...,NaN,NaN,NaN,https://doi.org/10.1016/j.bmcl.2025.130193,SKUSCUALKIOIST,NaN,NaN,NaN,NaN,NaN
2,920.0,825.0,Pomalidomide,Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O,UVSMNLNDYGZFPF-UHFFFAOYSA-N,InChI=1S/C13H11N3O4/c14-7-3-1-2-6-10(7)13(20)1...,UVSMNLNDYGZFPF-UHFFFAOYSA-N,PLZF,ZBTB16,Q05516,...,NaN,NaN,NaN,https://doi.org/10.1038/s42003-021-02801-y,UVSMNLNDYGZFPF,NaN,NaN,NaN,NaN,NaN
3,1215.0,1049.0,TMX-4100,O=C1CCC(n2cc(-c3ccsc3)nn2)C(=O)N1,PGEUBQZQGXWTFO-UHFFFAOYSA-N,InChI=1S/C11H10N4O2S/c16-10-2-1-9(11(17)12-10)...,PGEUBQZQGXWTFO-UHFFFAOYSA-N,PDE6D,PDE6D,O43924,...,NaN,NaN,NaN,https://doi.org/10.1021/acs.jmedchem.1c01832,PGEUBQZQGXWTFO,NaN,NaN,NaN,NaN,NaN
4,1463.0,1263.0,LC-03-041,O=C1CC[C@H](N2Cc3cc(-c4cnc5n4CCNC5)ccc3C2=O)C(...,WDUXNRRGWRVGDQ-AWEZNQCLSA-N,InChI=1S/C19H19N5O3/c25-17-4-3-14(18(26)22-17)...,WDUXNRRGWRVGDQ-AWEZNQCLSA-N,NEK7,NEK7,Q8TDX7,...,NaN,NaN,NaN,https://doi.org/10.1002/anie.202500169,WDUXNRRGWRVGDQ,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
613,NaN,NaN,lenalidomide,Nc1cccc2c1CN(C1CCC(=O)NC1=O)C2=O,GOTYRUGSSMKFNF-UHFFFAOYSA-N,NaN,GOTYRUGSSMKFNF-UHFFFAOYSA-N,CK1α,CSNK1A1,P48729,...,NaN,NaN,NaN,10.1186/s13045-024-01592-z,GOTYRUGSSMKFNF,AUTOTPD-MG-000242,3-(7-amino-3-oxo-1H-isoindol-2-yl)piperidine-2...,PUBCHEM,PUBCHEM,PUBCHEM
614,NaN,NaN,pomalidomide,Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O,UVSMNLNDYGZFPF-UHFFFAOYSA-N,NaN,UVSMNLNDYGZFPF-UHFFFAOYSA-N,CK1α,CSNK1A1,P48729,...,NaN,NaN,NaN,10.1186/s13045-024-01592-z,UVSMNLNDYGZFPF,AUTOTPD-MG-000243,"4-amino-2-(2,6-dioxopiperidin-3-yl)isoindole-1...",PUBCHEM,PUBCHEM,PUBCHEM
615,NaN,NaN,pomalidomide,Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O,UVSMNLNDYGZFPF-UHFFFAOYSA-N,NaN,UVSMNLNDYGZFPF-UHFFFAOYSA-N,IKZF1,IKZF1,Q13422,...,NaN,NaN,NaN,10.1186/s13045-024-01592-z,UVSMNLNDYGZFPF,AUTOTPD-MG-000244,"4-amino-2-(2,6-dioxopiperidin-3-yl)isoindole-1...",PUBCHEM,PUBCHEM,PUBCHEM
616,NaN,NaN,pomalidomide,Nc1cccc2c1C(=O)N(C1CCC(=O)NC1=O)C2=O,UVSMNLNDYGZFPF-UHFFFAOYSA-N,NaN,UVSMNLNDYGZFPF-UHFFFAOYSA-N,IKZF2,IKZF2,Q9UKS7,...,NaN,NaN,NaN,10.1186/s13045-024-01592-z,UVSMNLNDYGZFPF,AUTOTPD-MG-000245,"4-amino-2-(2,6-dioxopiperidin-3-yl)isoindole-1...",PUBCHEM,PUBCHEM,PUBCHEM


## Continue the AutoTPD_ID series onto the database rows


In [4]:
def extend_autotpd_id(merged_df, id_col='AutoTPD_ID'):
    """Continue the AutoTPD_ID series onto the rows that came from the source database.

    Rows carried over from AutoTPD keep the id they were given in
    data/release/AutoTPD-*.csv; the database rows are numbered from the next free
    number in the same series, so one id always denotes one record. The prefix and
    zero-padding are read off the existing ids rather than hard-coded.
    """
    ids = merged_df[id_col].astype('string')
    existing = ids.dropna()
    if existing.empty:
        raise ValueError(f'no {id_col} values to continue from - re-run '
                         'data_standardization.py with --record_id_prefix')

    prefix = existing.iloc[0].rsplit('-', 1)[0]
    width = len(existing.iloc[0].rsplit('-', 1)[1])
    next_n = existing.str.rsplit('-', n=1).str[-1].astype(int).max() + 1

    missing = ids.isna()
    merged_df.loc[missing, id_col] = [
        f'{prefix}-{n:0{width}d}' for n in range(next_n, next_n + int(missing.sum()))
    ]

    merged_df = merged_df.sort_values(id_col, ignore_index=True)
    merged_df = merged_df[[id_col] + [c for c in merged_df.columns if c != id_col]]

    print(f'{id_col}: {len(existing)} carried over from AutoTPD, '
          f'{int(missing.sum())} newly assigned to database rows')
    print(f'  range {merged_df[id_col].iloc[0]} .. {merged_df[id_col].iloc[-1]}, '
          f'unique={merged_df[id_col].is_unique}, no gaps={len(merged_df) == int(merged_df[id_col].str.rsplit("-", n=1).str[-1].astype(int).max())}')
    return merged_df


df_protac_merged = extend_autotpd_id(df_protac_merged)
df_mg_merged = extend_autotpd_id(df_mg_merged)


AutoTPD_ID: 714 carried over from AutoTPD, 1818 newly assigned to database rows
  range AUTOTPD-PROTAC-000001 .. AUTOTPD-PROTAC-002532, unique=True, no gaps=True
AutoTPD_ID: 246 carried over from AutoTPD, 372 newly assigned to database rows
  range AUTOTPD-MG-000001 .. AUTOTPD-MG-000618, unique=True, no gaps=True


In [5]:
df_protac_merged.to_csv('/Users/yaochenr/project/tpd_curator/data/release/AutoTPDplus-PROTAC.csv', index=False)
df_mg_merged.to_csv('/Users/yaochenr/project/tpd_curator/data/release/AutoTPDplus-MG.csv', index=False)